### Pipeline

<pre>
Capture frame
    ↓
Convert BGR → RGB
    ↓
Detect faces
    ↓
Extract embeddings
    ↓
Match against database
    ↓
Draw boxes + labels
    ↓
Show frame
    ↓
  Repeat
</pre>

### Step 1 — Capturing Frame and Converting it to PIL RGB

- Cv2 captures BRG

- Both PIL and MTCNN expect RGB

So we must convert.

Why This Matters

If you skip conversion:

- Faces may not detect properly
- Embeddings may be wrong

Because red and blue channels are swapped.

In [8]:
import torch
import cv2
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import os
import torch.nn.functional as F
import time

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(image_size=160, margin=0, device=device, keep_all=True)

resnet = InceptionResnetV1(pretrained="vggface2").eval().to(device)

In [10]:
database = {}

for person in os.listdir('faces/single'):
    person_embeddings = []
    
    for file in os.listdir(f'faces/single/{person}'):
        img = Image.open(os.path.join(f'faces/single/{person}', file))
        
        face = mtcnn(img)
        
        if face is None:
            continue
        
        # If multiple faces are detected take first one:
        if face.ndim == 4:
            face = face[0].unsqueeze(0)
        
        face = face.to(device)
        
        with torch.no_grad():
            embedding = resnet(face)
            
        # Normalize embedding
        embedding /= embedding.norm(dim=1, keepdim=True)
        
        person_embeddings.append(embedding)
        
    if len(person_embeddings) > 0:
        # Average embeddings
        database[person] = torch.mean(torch.cat(person_embeddings), dim=0, keepdim=True)
            
print("Database Loaded: ", list(database.keys()))

Database Loaded:  ['abhishek_bachchan', 'amitabh_bachchan', 'chirag', 'jaya_bachchan', 'shail']


In [ ]:
def recognize_face(frame, database, frame_count, stored_boxes, stored_labels, threshold=0.8):
    
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(frame_rgb)
    
    # detect every N frames
    if frame_count % 5 == 0:
        boxes, _ = mtcnn.detect(img)

        if boxes is None:
            time.sleep(0.5)
            return frame, None, None
        
        faces = mtcnn(img)
        
        if faces is None:
            time.sleep(0.5)
            return frame, None, None
        
        if faces.ndim == 3:
            faces = faces.unsqueeze(0)
    
        faces = faces.to(device)      
        
        
        with torch.no_grad():
            embeddings = resnet(faces)
        
        # Normalize embeddings
        embeddings /= embeddings.norm(dim=1, keepdim=True)
        
        labels = []
    
        for query_embedding in embeddings:
            
            query_embedding = query_embedding.unsqueeze(0)
            
            best_match = None
            max_similarity = -1
        
            for name, db_embedding in database.items():
                
                similarity = F.cosine_similarity(
                    query_embedding,
                    db_embedding
                ).item()
                
                if similarity > max_similarity:
                    max_similarity = similarity
                    best_match = name
                    
            #Decide label
            if max_similarity > threshold:
                labels.append(best_match)
            else:
                labels.append("Unknown")
            
        stored_boxes = boxes
        stored_labels = labels
        
    else:
        boxes = stored_boxes
        labels = stored_labels
            
    if boxes is None:
        return frame, None, None
    
    # draw        
    for i, box in enumerate(boxes):
        
        x1, y1, x2, y2 = map(int, box)
        
        label = labels[i] if i < len(labels) else "Unknown"
        
                    
        color = (0, 0, 255) if label == "Unknown" else (0, 255, 0)
        
        # text_y = y1 - 10 if y1 - 10 > 10 else y1 + 20
        
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
        # (text_width, text_height), _ = cv2.getTextSize(
        #     label,
        #     cv2.FONT_HERSHEY_SIMPLEX,
        #     0.6,
        #     2
        # )
        
        # cv2.rectangle(
        #     frame,
        #     (x1, text_y - text_height - 5),
        #     (x2 + text_width, text_y + 5),
        #     color,
        #     -1
        # )
        
        # cv2.putText(
        #     frame,
        #     label,
        #     (x1, text_y),
        #     cv2.FONT_HERSHEY_SIMPLEX,
        #     0.6,
        #     (0, 0, 0),
        #     2
        # )
        
    return frame, stored_boxes, stored_labels      
        

In [12]:
cap = cv2.VideoCapture(0)
prev_time = time.time()
frame_count = 0
stored_boxes = None
stored_labels = None
display_fps = 0
fps = 0

while True:
    ret, frame = cap.read()
    frame = cv2.resize(frame, (640, 480))
    
    if not ret:
        break
    
    frame_count += 1
    
    frame, stored_boxes, stored_labels = recognize_face(frame, database, frame_count, stored_boxes, stored_labels)
    
    # FPS
    current_time = time.time()
    fps = 0.9 * fps + 0.1 * (1 / (current_time - prev_time))
    prev_time = current_time

    if(frame_count % 5 == 0):
        display_fps = f"FPS: {int(fps)}"
        
    if(stored_boxes is None and stored_labels is None):
        display_fps = "Idle Mode"

    cv2.putText(
        frame,
        display_fps,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )
    
    cv2.imshow("Webcam", frame)
    
    if cv2.waitKey(1) &0xFF == ord('q'):
        break
    
cap.release()
cv2.destroyAllWindows()